# Georeferenciamento de Imagens

[Image-to-Image Georeferencing using python](https://freedium.cfd/medium.com/@adityakuche/image-to-image-georeferencing-using-python-78351259bc8f)


Zuado!

In [ ]:
import os
import pprint
import shutil
import tempfile
from datetime import date, datetime
from pathlib import Path

import geopandas as gpd
import rasterio
import rasterio.plot
from cbers4asat import Cbers4aAPI
from cbers4asat import Collections as coll
from cbers4asat import tools
from dotenv import load_dotenv
from osgeo import gdal, ogr, osr
from osgeo_utils import gdal_pansharpen

import open_geodata as geo

<br>

-----

## Cria Pasta Temporária

In [ ]:
# Crio pasta temporária
temp_path = Path(tempfile.gettempdir()) / 'open_geodata' / 'raster' / 'cbers'
temp_path.mkdir(exist_ok=True)


In [ ]:
list_dirs = [x for x in list(temp_path.glob('*')) if x.is_dir()]
list_dirs

In [ ]:
# Processamento L4: Georreferenciada
imagem_l4 = temp_path / 'CBERS4A_WPM23312520250706ETC2' / 'pansharp_gdal.tif'

# Processamento L2: para georreferenciar
imagem_l2 = temp_path / 'CBERS4A_WPM23312520250505ETC2' / 'pansharp_gdal.tif'
imagem_l2_out = temp_path / 'CBERS4A_WPM23312520250505ETC2' / 'pansharp_gdal_georef.tif'

shutil.copy(imagem_l4, imagem_l2_out) 

<br>

-----

## Abre Imagens

In [ ]:
# 
in_raster = gdal.Open(str(imagem_l4))
out_raster=gdal.Open(str(imagem_l2_out), gdal.GA_Update)

In [ ]:
# Obtém o sistema de referência espacial da imagem de referência
sr = osr.SpatialReference()

# My projection system
# sr.ImportFromEPSG(32634)
sr.ImportFromWkt(in_raster.GetProjection())

<br>

-----

## Método 1

In [ ]:
# gdal.GCP(a,b,0,i,j)
def Boundbox(dataset):    
    # Get the image's geotransform
    geotransform = dataset.GetGeoTransform()

    # Get the dimensions of the image
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    print(f'Pixel size {width}')

    # Calculate the coordinates of the corners
    top_left_x = geotransform[0]
    top_left_y = geotransform[3]
    top_right_x = geotransform[0] + width * geotransform[1]
    top_right_y = geotransform[3]
    bottom_left_x = geotransform[0]
    bottom_left_y = geotransform[3] + height * geotransform[5]
    bottom_right_x = geotransform[0] + width * geotransform[1]
    bottom_right_y = geotransform[3] + height * geotransform[5]

    return (
        top_left_x,
        top_left_y,
        top_right_x,
        top_right_y,
        bottom_left_x,
        bottom_left_y,
        bottom_right_x,
        bottom_right_y,
    )


In [ ]:
in_bb = Boundbox(dataset=in_raster)  # calling the function
out_bb = Boundbox(dataset=out_raster)  # getting the corrdenated value

# separating the value
a, b, c, d, e, f, g, h = in_bb
i, j, k, l, m, n, o, p = out_bb

# Getting the value to the GCP's (Ground Control Points) one image respect to another images
gcps = [
    gdal.GCP(a, b, 0, i, j),
    gdal.GCP(c, d, 0, k, l),
    gdal.GCP(e, f, 0, m, n),
    gdal.GCP(g, h, 0, o, p),
]

gcps

In [ ]:
out_raster.SetGCPs(gcps, sr.ExportToWkt()) #setting the projection and GCP's(image to image reference)
out_raster= None #closing the file

<br>

-----

## Método 2

In [ ]:
# Função para pegar os cantos da imagem
def Boundbox(dataset):
    geotransform = dataset.GetGeoTransform()
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    return [
        (geotransform[0], geotransform[3]),  # top-left
        (geotransform[0] + width * geotransform[1], geotransform[3]),  # top-right
        (geotransform[0], geotransform[3] + height * geotransform[5]),  # bottom-left
        (geotransform[0] + width * geotransform[1], geotransform[3] + height * geotransform[5])  # bottom-right
    ]


In [ ]:
# Pega os cantos das duas imagens
in_corners = Boundbox(in_raster)
out_corners = Boundbox(out_raster)

In [ ]:
# Cria os GCPs (Ground Control Points)
gcps = [
    gdal.GCP(in_corners[0][0], in_corners[0][1], 0, 0, 0),  # top-left
    gdal.GCP(in_corners[1][0], in_corners[1][1], 0, out_raster.RasterXSize, 0),  # top-right
    gdal.GCP(in_corners[2][0], in_corners[2][1], 0, 0, out_raster.RasterYSize),  # bottom-left
    gdal.GCP(in_corners[3][0], in_corners[3][1], 0, out_raster.RasterXSize, out_raster.RasterYSize)  # bottom-right
]

# Aplica os GCPs e o sistema de referência espacial
out_raster.SetGCPs(gcps, sr.ExportToWkt())
out_raster = None  # Fecha o arquivo